In [ ]:
# Import libraries
import requests
import json
import os
from datetime import datetime, timedelta

In [ ]:
# Define parameters that will come from pipeline during execution

## Pipeline will define end_date as base parameter for all activities in the pipeline using @utcNow()
end_date = None

## The next parameters will come from bronze_control table in metadata warehouse, retrieved using pipeline activities
esios_indicator_id = None
esios_indicator_short_name = None
locale_language = None
time_trunc = None
time_agg = None
target_lakehouse_path = None
target_landing_path = None
target_folder_name = None
file_type = None
last_watermark = None
overlap_minutes = None

In [ ]:
# Define other variables for API call

## Start and end dates in ISO8601 format with time zone and considering overlap minutes for start date
overlap_minutes = int(overlap_minutes)
end_date_dt = datetime.fromisoformat(end_date.replace("Z", "+00:00"))
start_date_dt = datetime.fromisoformat(last_watermark.replace("Z", "+00:00")) - timedelta(minutes=overlap_minutes)

start_date = start_date_dt.strftime("%Y-%m-%dT%H:%M:%SZ")
end_date = end_date_dt.strftime("%Y-%m-%dT%H:%M:%SZ")

## Execution date for file name
execution_date = end_date_dt.strftime("%Y-%m-%d_%H-%M-%S")

## Other variables for the request
url = f"https://api.esios.ree.es/indicators/{esios_indicator_id}"
headers = {'Accept': 'application/json; application/vnd.esios-api-v1+json',
           'Content-Type': 'application/json',
           'x-api-key': f'{api_key}'}
params = {
    "locale": f"{locale_language}",
    "start_date": f"{start_date}",
    "end_date": f"{end_date}",
    "time_trunc": f"{time_trunc}",
    "time_agg": f"{time_agg}"
} 

In [ ]:
# Make API call
try:
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    data = response.json()
    if not data:
        raise ValueError(f"No data returned for indicator {esios_indicator_id} between {start_date} and {end_date}")
except requests.exceptions.RequestException as e:
    raise RuntimeError(f"Error fetching data from API: {e}")

# Define paths
file_path = f"{target_lakehouse_path}/{target_landing_path}/{target_folder_name}/indicator_{esios_indicator_id}_{execution_date}.{file_type}"
directory_path = f"{target_lakehouse_path}/{target_landing_path}/{target_folder_name}"

# Create folder if it doesnt exists and save json file 
try:
    notebookutils.fs.mkdirs(directory_path)
    notebookutils.fs.put(file_path, json.dumps(data, indent=4), True)
    print(f"Data successfully saved to {file_path}")
except Exception as e:
    raise RuntimeError(f"Error saving file to {file_path}. Details: {e}")